# Dispersion on a uniform chain: particles against the analytical solution

A synthetic river of 100 identical reaches with steady flow, one slug released near the upstream end,
and the resulting concentration compared with the closed-form solution of the advection-dispersion
equation. Because every reach has the same velocity and Fischer coefficient, the chain behaves like one
long channel, so the particle results should reproduce the Gaussian plume exactly up to sampling noise.

In [ ]:
import pathlib
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from scipy import stats

from fluvial_particle import FileHydraulicsProvider, NetworkConfig, run_network_simulation
from fluvial_particle.network.dispersion import fischer_coefficient

OUT = pathlib.Path("./network-chain-dispersion-output")
OUT.mkdir(exist_ok=True)

# channel and flow
N_REACH, LENGTH = 100, 1000.0          # 100 reaches of 1 km
VELOCITY, DEPTH, WIDTH, SLOPE = 0.5, 1.0, 20.0, 1e-4
DAYS = 3

# release
MASS, X0 = 1000.0, 500.0               # kg; 500 m into the first reach, clear of the headwater boundary
N_PARTICLES = 50_000
DT, OUTPUT_INTERVAL = 300.0, 1800.0

import textwrap

def finish(fig, name, caption):
    """Add a caption under the figure, save it to OUT as a 150 dpi PNG for sharing, and show it."""
    fig.text(0.5, -0.02, textwrap.fill(caption, 125), ha="center", va="top", fontsize=9.5, wrap=True)
    path = OUT / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path}")
    plt.show()

## 1. A synthetic hydraulics file

The solver reads the network hydraulics NetCDF schema written by pywatershed. For a uniform chain the file
is easy to build by hand: reach *i* flows into reach *i + 1*, the last reach is the outlet, and every
time-varying field is constant. Shear velocity follows the export's definition, `sqrt(g * depth * slope)`.

In [ ]:
def write_chain_hydraulics(path, n_reach, length, velocity, depth, width, slope, days):
    """Write a uniform-chain hydraulics file in the pywatershed network export schema."""
    n = n_reach
    times = np.datetime64("2000-01-01", "ns") + np.arange(days + 1) * np.timedelta64(1, "D")
    nt = times.size
    reach_id = np.arange(1, n + 1, dtype=np.int64)
    to_index = np.arange(1, n + 1, dtype=np.int32)
    to_index[-1] = -1
    ustar = np.sqrt(9.80665 * depth * slope)
    flow = velocity * depth * width

    def static(values, units, dtype=None):
        return xr.DataArray(np.asarray(values, dtype=dtype), dims=("reach",), attrs={"units": units})

    def field(value, units):
        return xr.DataArray(np.full((nt, n), float(value)), dims=("time", "reach"), attrs={"units": units})

    ds = xr.Dataset(
        {
            "reach_id": static(reach_id, "-", np.int64),
            "to_id": static(np.where(to_index >= 0, reach_id[np.clip(to_index, 0, n - 1)], 0), "-", np.int64),
            "to_index": static(to_index, "-", np.int32),
            "is_outlet": static((to_index < 0).astype(np.int8), "-"),
            "length": static(np.full(n, length), "m"),
            "slope": static(np.full(n, slope), "m m-1"),
            "mann_n": static(np.full(n, 0.035), "s m-1/3"),
            "elevation_mid": static(100.0 - slope * (np.arange(n) + 0.5) * length, "m"),
            "bankfull_width": static(np.full(n, width), "m"),
            "bankfull_depth": static(np.full(n, depth), "m"),
            "x_mid": static((np.arange(n) + 0.5) * length, "m"),
            "y_mid": static(np.zeros(n), "m"),
            "flow_in": field(flow, "m3 s-1"),
            "flow_out": field(flow, "m3 s-1"),
            "velocity": field(velocity, "m s-1"),
            "depth": field(depth, "m"),
            "width": field(width, "m"),
            "ustar": field(ustar, "m s-1"),
            "residence_time": field(length / velocity, "s"),
        },
        coords={"time": times},
    )
    ds.attrs = {
        "title": "synthetic uniform chain", "source_model": "synthetic", "n_unconnected": -1, "connect_tol": 1.0,
        "crs_wkt": "", "conventions_note": "Particle state is (reach index, s) with 0 <= s <= length from the "
        "reach's upstream end; to_index == -1 is an outlet.",
    }
    ds.to_netcdf(path, engine="h5netcdf")
    return path

HYDRAULICS = write_chain_hydraulics(OUT / "chain_hydraulics.nc", N_REACH, LENGTH, VELOCITY, DEPTH, WIDTH, SLOPE, DAYS)

with FileHydraulicsProvider(HYDRAULICS) as prov:
    h = prov.hydraulics(prov.times[0])
    K = float(fischer_coefficient(h["velocity"], h["depth"], h["width"], h["ustar"])[0])
AREA = WIDTH * DEPTH
print(f"Fischer dispersion coefficient K = 0.011 v^2 w^2 / (d u*) = {K:.1f} m2/s; "
      f"per-step kick sqrt(2 K dt) = {np.sqrt(2 * K * DT):.0f} m against {VELOCITY * DT:.0f} m of advection")

## 2. Run: one slug, three days

The slug sits 500 m into reach 1 so the plume's early tail does not touch the headwater boundary, where an
upstream kick reflects. With `particles` set on the source, every particle carries the same mass, here 20 g.

In [ ]:
cfg = NetworkConfig.from_dict({
    "hydraulics_file": str(HYDRAULICS),
    "start_time": "2000-01-01", "end_time": f"2000-01-0{1 + DAYS}",
    "dt": DT, "output_interval": OUTPUT_INTERVAL, "mass_units": "kg",
    "dispersion": {"model": "fischer"},
    "sources": [{"reach_id": 1, "form": "slug", "time": 0.0, "mass": MASS, "s": X0, "particles": N_PARTICLES}],
    "seed": 7,
})
res = run_network_simulation(cfg, OUT)
print(res.summary())
t_sec = res.time_seconds

## 3. The analytical solution

For an instantaneous release of mass *M* at *x₀* into a uniform channel of cross-section *A* with velocity *v*
and dispersion coefficient *K*, the advection-dispersion equation gives

$$C(x, t) = \frac{M}{A\sqrt{4\pi K t}} \exp\left(-\frac{(x - x_0 - v t)^2}{4 K t}\right),$$

a Gaussian with mean *x₀ + vt* and variance *2Kt*. The solver's per-step random kick has exactly that
variance, so the comparison tests the whole chain of bookkeeping: advection with time carry across 100
reach boundaries, the kick, and the binning in post-processing.

In [ ]:
def analytical(x, t):
    """Slug solution of the advection-dispersion equation (kg/m3) at distance x (m) and time t (s)."""
    x, t = np.asarray(x, dtype=float), np.asarray(t, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        c = MASS / (AREA * np.sqrt(4 * np.pi * K * t)) * np.exp(-((x - X0 - VELOCITY * t) ** 2) / (4 * K * t))
    return np.where(t > 0, c, 0.0)

bins = res.bins(500.0)
x_bins = bins.bin_reach * LENGTH + 0.5 * (bins.s_start + bins.s_end)   # distance of each bin centre from the chain's head
cube = res.concentration(None, bin_length=500.0).values                # (time, bin) kg/m3

## 4. Concentration along the channel at four times

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
x_fine = np.linspace(0, N_REACH * LENGTH, 2000)
for day in (0.5, 1.0, 1.5, 2.0):
    i = int(np.argmin(np.abs(t_sec - day * 86400)))
    color = ax.plot(x_fine / 1000, analytical(x_fine, t_sec[i]), lw=1.5, label=f"analytical, t = {day:g} d")[0].get_color()
    ax.plot(x_bins / 1000, cube[i], ".", ms=4, color=color, alpha=0.8, label=f"particles, t = {day:g} d")
ax.set_xlabel("distance along the chain (km)")
ax.set_ylabel("concentration (kg/m3)")
ax.set_title(f"Slug plume along a uniform chain: {N_PARTICLES:,} particles in 500 m bins against the Gaussian solution")
ax.legend(ncol=2, title=f"v = {VELOCITY} m/s, K = {K:.1f} m2/s (Fischer), M = {MASS:g} kg, dt = {DT:g} s")
finish(fig, "fig1_plume_snapshots", (
    f"Figure 1. Concentration along a {N_REACH} x {LENGTH / 1000:g} km chain at four times after a {MASS:g} kg slug "
    f"released {X0:g} m from the upstream end. Points are particle mass per 500 m bin divided by the bin volume "
    f"(width {WIDTH:g} m x depth {DEPTH:g} m x 500 m); lines are the analytical slug solution of the advection-dispersion "
    f"equation with the same velocity and Fischer coefficient K = 0.011 v^2 w^2 / (d u*). The plume crosses about "
    f"{VELOCITY * 86400 / LENGTH:.0f} reach boundaries per day; the agreement shows that the time carry across "
    f"boundaries and the per-step dispersive kick reproduce the continuous solution."))

## 5. Concentration versus time at three stations

In [ ]:
stations_km = (20, 50, 80)
fig, ax = plt.subplots(figsize=(11, 5))
for x_km in stations_km:
    reach = int(x_km * 1000 // LENGTH)
    b = int(bins.bin_of(np.array([reach]), np.array([x_km * 1000 - reach * LENGTH]))[0])
    color = ax.plot(t_sec / 86400, analytical(x_bins[b], t_sec), lw=1.5, label=f"analytical, {x_km} km")[0].get_color()
    ax.plot(t_sec / 86400, cube[:, b], ".", ms=4, color=color, alpha=0.8, label=f"particles, {x_km} km")
ax.set_xlabel("time (days)")
ax.set_ylabel("concentration (kg/m3)")
ax.set_title("Breakthrough curves at fixed stations")
ax.legend(ncol=3, title=f"v = {VELOCITY} m/s, K = {K:.1f} m2/s, 500 m bins, output every {OUTPUT_INTERVAL / 60:g} min")
finish(fig, "fig2_station_breakthrough", (
    f"Figure 2. Concentration versus time at stations {', '.join(str(s) for s in stations_km)} km downstream of the head of "
    f"the chain (points: particles in the 500 m bin containing the station; lines: analytical). Peak concentration falls "
    f"and the pulse widens with distance as the plume variance grows as 2Kt; the arrival of the peak is x / v."))

## 6. Plume variance versus time

The second moment isolates the dispersion bookkeeping from everything else. Only output times before the
first exit are used, so the sample is the whole plume.

In [ ]:
ds = res.positions()
status = ds["status"].values
dist = ds["reach_index"].values * LENGTH + ds["s"].values      # NaN where inactive
whole = ((status == 2).sum(axis=1) == 0) & ((status == 1).sum(axis=1) > 0)   # released, and no exits yet
var_particles = np.nanvar(dist[whole], axis=1)
mean_particles = np.nanmean(dist[whole], axis=1)
tw = t_sec[whole]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(tw / 86400, 2 * K * tw / 1e6, lw=1.5, label="2 K t")
axes[0].plot(tw / 86400, var_particles / 1e6, ".", ms=4, label="particles")
axes[0].set_ylabel("plume variance (km2)")
axes[1].plot(tw / 86400, (X0 + VELOCITY * tw) / 1000, lw=1.5, label="x0 + v t")
axes[1].plot(tw / 86400, mean_particles / 1000, ".", ms=4, label="particles")
axes[1].set_ylabel("plume centre (km)")
axes[0].set_title("Plume variance"); axes[1].set_title("Plume centre")
for ax in axes:
    ax.set_xlabel("time (days)"); ax.legend(title=f"{N_PARTICLES:,} particles, K = {K:.1f} m2/s, v = {VELOCITY} m/s")
finish(fig, "fig3_plume_moments", (
    "Figure 3. Second and first moments of the particle cloud against theory, using every output time before the "
    "first particle exits so the whole plume is sampled. The variance is the direct check of the dispersion "
    "bookkeeping (each step adds exactly 2 K dt); the centre is the check of exact advection with time carry."))
i = int(np.nonzero(whole)[0][-1])
print(f"at t = {tw[-1] / 86400:.2f} d: variance {var_particles[-1] / 1e6:.3f} km2 vs 2Kt {2 * K * tw[-1] / 1e6:.3f} km2; "
      f"centre {mean_particles[-1] / 1000:.2f} km vs {(X0 + VELOCITY * tw[-1]) / 1000:.2f} km")

## 7. Arrival times at the outlet

The first-passage time of a particle to the end of the chain is inverse-Gaussian with mean *L/v* and shape
*L²/2K*, where *L* is the distance from the release to the outlet. The particle arrivals sit slightly late
because the outlet is checked only at the end of each step; the spec bounds that bias by
0.5826·sqrt(2 K dt)/v + dt/2, which is a few minutes here against a spread of hours.

In [ ]:
arr = res.arrival_times()
L = N_REACH * LENGTH - X0
mean_t, shape = L / VELOCITY, L**2 / (2 * K)
ig = stats.invgauss(mean_t / shape, scale=shape)
edges = np.linspace(arr.exit_time.min(), arr.exit_time.max(), 80)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(arr.exit_time / 3600, bins=edges / 3600, density=True, alpha=0.5, label="particle arrivals")
tt = np.linspace(edges[0], edges[-1], 800)
ax.plot(tt / 3600, ig.pdf(tt) * 3600, lw=1.5, label="inverse Gaussian")
ax.set_xlabel("arrival time at the outlet (h)"); ax.set_ylabel("density (1/h)")
ax.set_title(f"First-passage times to the outlet, {L / 1000:g} km from the release")
ax.legend(title=f"inverse Gaussian: mean L/v = {mean_t / 3600:.1f} h, shape L^2/2K")
finish(fig, "fig4_outlet_arrivals", (
    f"Figure 4. Distribution of particle arrival times at the outlet (histogram, {N_PARTICLES:,} particles) against the "
    f"inverse-Gaussian first-passage distribution of Brownian motion with drift. Particles arrive slightly late because "
    f"the outlet is checked only at step ends; the bias is bounded by 0.5826 sqrt(2 K dt)/v + dt/2 = "
    f"{(0.5826 * np.sqrt(2 * K * DT) / VELOCITY + DT / 2) / 60:.1f} min here, small against the {np.sqrt(mean_t**3 / shape) / 3600:.1f} h spread."))
bound = 0.5826 * np.sqrt(2 * K * DT) / VELOCITY + DT / 2
print(f"{len(arr)} of {N_PARTICLES} particles exited; mean arrival {arr.exit_time.mean() / 3600:.2f} h vs {mean_t / 3600:.2f} h "
      f"analytical (late by {(arr.exit_time.mean() - mean_t) / 60:.1f} min; documented bound {bound / 60:.1f} min); "
      f"std {arr.exit_time.std() / 3600:.2f} h vs {np.sqrt(mean_t**3 / shape) / 3600:.2f} h")

In [ ]:
res.close()